# AI4GSD Demo
------------------------------------------
## Smartphone photos and videos
------------------------------------------
#### Table of contents
#### 0. Installing requried packages
#### 1. Importing required packages
#### 2. Defining working directory
#### 3. Extracting or copying your videos/photos to desired folder
#### 4. Obtaining scales from AI, drone flight data, or manual measurements
#### 5. Obtaining segmentation of anthropogenic objects
#### 6. Obtaining grain size data
#### 7. Applying quality control

##### *** Please contact Yunxiang Chen, yunxiang.chen@pnnl.gov for reporting errors

------------------------------------------
### 0. Installing requried packages

#### 0a. Mount workshop share drive to your personal google drive.

In [1]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


### 0b. Install requried packages

In [2]:
%pip install -q numpy==2.2.6 pandas==2.2.3 matplotlib==3.10.6 opencv-python==4.12.0.88
%pip install -q torch torchvision ultralytics==8.3.186
%pip install -q natsort pyproj PyExifTool piexif imutils shapely psutil

### 0c. Define Google drive parent path and add it to system.

In [3]:
import sys
from pathlib import Path
WorkDir = Path("/content/drive/MyDrive/AI4GSD")
sys.path.insert(0,str(WorkDir))

exiftool = WorkDir/"AI4GSD/exes/exiftool_linux/Image-ExifTool-13.34/exiftool" # Enable wrting permission in Google drive.
exiftool.chmod(0o755)

### 1. Importing required packages

In [4]:
import AI4GSD

### 2. Defining working directory

In [5]:
AI4GSD.pcinfo()
foldername = '20250627_LR_WalkSmartphoneFieldTest'                         # Parent folder where you save photos/videos.
PhotoDir = str(WorkDir / 'Demo_Data' / foldername)                         # AI4GSD use string path.

Machine name: aa520db17c17
Operating system: Ubuntu 24.04.5 LTS
Architecture: x86_64
Processor: Intel(R) Xeon(R) CPU @ 2.20GHz
Reported CPU frequency: 2.20 GHz
PyTorch version: 2.14.1+cpu
CUDA available: False
CUDA GPU unavailable to PyTorch


### 3. Extracting or copying your videos/photos to desired folder

In [6]:
AI4GSD.getPhotos(PhotoDir,nframe=150, overwrite=False)                     # Use once.

Video photo output folder: /content/drive/MyDrive/AI4GSD/Demo_Data/20250627_LR_WalkSmartphoneFieldTest
Extracting IMG_7918.MOV every 150 frames.
IMG_7918.MOV: get frame 1 from 150/444 in 0.00 s (33.78%), file existed, skipping...
IMG_7918.MOV: get frame 2 from 300/444 in 0.00 s (67.57%), file existed, skipping...

Computation done, elapsed time 1.74 seconds.




### 4. Obtaining scales from AI, drone flight data, or manual measurements

In [7]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.OverWriteAll = True                                                     # Overwrite everything if True.
PP.OverWriteScaleFile = False
PP.OverWriteScaleRows = True
PP.ScaleSource = 'AI'                                                      # Use AI option.
PP.ModelName = 'YOLO11m.1280.20250825'                                     # Which AI to use.
PP.ComputingSource = 'local'                                               # What computing resource to use.
PP.ImageSizeFactor = 1                                                     # Do you need to user a larger image size for inference?
PP.ConfidenceThresholdUser = 0.85                                          # Minimum confidence threshold.
PP.YOLOSave = False                                                        # Need to save AI prediction to folder.
scales =  AI4GSD.metadata(PP)

Generating scale file for: 20250627_LR_WalkSmartphoneFieldTest
Estimating photo resolution via local AI model: YOLO11m.1280.20250825
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model layer: 113, model parmeter: 20.06M, submodule number: 409
IOU: 0.45, user confidence: 0.85
1/6: photo 1/6, IMG_7900.JPG, scale redbluecap, res 0.16 mm/px, status 200, (16.67%), 15.35 s
2/6: photo 2/6, IMG_7908.JPG, scale redbluecap, res 0.15 mm/px, status 200, (33.33%), 5.96 s
3/6: photo 3/6, IMG_7914.JPG, scale redbluecap, res 0.16 mm/px, status 200, (50.00%), 6.17 s
4/6: photo 4/6, IMG_7918_150.png, scale N/A, res 1 px/px, status 200, (66.67%), 8.70 s
5/6: photo 5/6, IMG_7918_300.png, scale redbluecap, res 0.35 mm/px, status 200, (83.33%), 3.

### 5. Obtaining segmentation of anthropogenic objects

In [8]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.FolderLevel = 'L1'
PP.ModelName = 'YOLO11m.640.20250906'
PP.ObjectName = 'anthro'
PP.OverWriteSegementation = True
PP.YOLOSave = False
PP.YOLOIOU = 0.45
PP.ComputingSource = 'local'
PP.ImageSizeType = 'default'
PP.ImageSizeFactor=1
PP.SegmentationClassName =['people','road','steel']                        # What object class need to track.
AI4GSD.background(PP)

-----------------------------------------------------------------
Folder name: 20250627_LR_WalkSmartphoneFieldTest
output name: segments
Model name: YOLO11m.640.20250906 from local for anthro
Model layer: 125, model parmeter: 22.37M, submodule number: 445
IOU: 0.45, user confidence: 0.35
Number of photos: 6, number of exluded photos 0
20250627_LR_WalkSmartphoneFieldTest: 1/6: IMG_7900.JPG (16.67%)
20250627_LR_WalkSmartphoneFieldTest: 2/6: IMG_7908.JPG (33.33%)
20250627_LR_WalkSmartphoneFieldTest: 3/6: IMG_7914.JPG (50.00%)
20250627_LR_WalkSmartphoneFieldTest: 4/6: IMG_7918_150.png (66.67%)
20250627_LR_WalkSmartphoneFieldTest: 5/6: IMG_7918_300.png (83.33%)
20250627_LR_WalkSmartphoneFieldTest: 6/6: IMG_7919.JPG (100.00%)
Total time: 28.98 s, time per photo 4.83 s
-----------------------------------------------------------------



### 6. Obtaining grain size data

In [9]:
PP = AI4GSD.parameters()
PP.Directory = PhotoDir
PP.ModelName = 'YOLO11m.1280.20250322'
PP.ImageSizeFactor = 1
PP.LineThickness = 2
PP.YOLOSave = False
PP.SaveFigure = True
PP.SaveOverlay = True
PP.OverWriteAll = True
PP.PrintOnScreen = True
PP.MaximumGrainSize = 0.2
gsd = AI4GSD.Photo2GSD(PP)

Output hidden; open in https://colab.research.google.com to view.

### 7. Applying quality control

In [10]:
gsdqc, stqc = AI4GSD.qualityControl(PhotoDir)


Quality control summary for 20250627_LR_WalkSmartphoneFieldTest:
1/1: IMG_7918.MOV, time 14.8 s, area 0.3 m2, valid area 0.3 m2 (100.0%), frame 2, valid frame 1 (50.0%)
1/1: ***, grain number 195, number per image 195.0; efficiency 0.0 m2/s, 13.2 grains/s
Photos: time 378.0 s, area 1.7 m2, valid area 1.4 m2 (82.9%), frame 4, valid frame 3 (75.0%)
Photos: grain number 794, number per image 198.5; efficiency 0.0 m2/s, 2.1 grains/s
All: time 392.8 s, area 2.0 m2, valid area 1.7 m2 (85.2%), frame 6, valid frame 4 (66.7%)
All: grain number 989, number per image 197.8; efficiency 0.0 m2/s, 2.5 grains/s
